In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import logging
from pathlib import Path

import os
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate
from omegaconf import OmegaConf

from pepo.utils import constants, set_seed

OmegaConf.register_new_resolver(
    "pepo.constants",
    lambda name: getattr(constants, name),
)


In [ ]:
config_path = Path("configs").absolute()
config_name = "eval.yaml"

with initialize_config_dir(config_dir=str(config_path), version_base="1.1"):
    cfg = compose(config_name=config_name)


original_work_dir = Path.cwd()

log_level_str = cfg.get("log_level", "INFO").upper()
log_level = getattr(logging, log_level_str, logging.INFO)

logger = instantiate(
    cfg.logger,
    log_dir=str(original_work_dir / "logs"),
    level=log_level,
)

resolved_cfg = OmegaConf.to_container(cfg, resolve=True)
logger.info("PEPO Evaluation - Starting")
logger.info(f"Configuration:\n{OmegaConf.to_yaml(resolved_cfg)}")

set_seed(cfg.seed)
logger.info(f"Random seed set to: {cfg.seed}")

if not os.getenv("HF_TOKEN"):
    logger.warning("HF_TOKEN environment variable not set. Model loading may fail if models are private.")

# Instantiate managers
device_manager = instantiate(cfg.device, logger=logger)
hub_manager = instantiate(cfg.hub, logger=logger)

# Instantiate model (same as chat.py)
model = instantiate(
    cfg.model,
    logger=logger,
    device_manager=device_manager,
    hub_manager=hub_manager,
)


In [ ]:
# Instantiate evaluator
evaluator = instantiate(cfg.evaluator, logger=logger, model=model)

responses_exist = evaluator.responses_exist()
force_regenerate = cfg.get("force_regenerate", False)
print(responses_exist, force_regenerate)

In [ ]:
evaluator._generate_filename()

In [ ]:
evaluator.dataset

In [ ]:
if not responses_exist or force_regenerate:
    output_ids, output_mask = evaluator.generate_responses()


In [ ]:
evaluator.evaluate()